In [1]:
import shutil
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import CrossEncoder

/tmp/ipykernel_74752/3652768135.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader


In [2]:
PDF_PATH = "./data"            
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "gemma4:31b-cloud"
CHROMA_DIR = "./chroma_db"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOP_K = 4                      

In [3]:
docs = DirectoryLoader(
    PDF_PATH, glob="**/*.pdf", loader_cls=PyPDFLoader, show_progress=True
).load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
)
chunks = splitter.split_documents(docs)

n_files = len({d.metadata["source"] for d in docs})
print(f"Loaded {n_files} PDFs, {len(docs)} pages, split into {len(chunks)} chunks.")

100%|██████████| 6/6 [00:00<00:00,  6.30it/s]

Loaded 6 PDFs, 289 pages, split into 157 chunks.


In [4]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

shutil.rmtree(CHROMA_DIR, ignore_errors=True)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
llm = ChatOllama(model=LLM_MODEL, temperature=0)

prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer the question using only the context below. "
    "If the context does not contain the answer, say you don't know.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

In [6]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(question, chunks, top_n=TOP_K):
    scores = reranker.predict([(question, c.page_content) for c in chunks])
    ranked = sorted(zip(scores, chunks), key=lambda x: x[0], reverse=True)
    return [c for _, c in ranked[:top_n]]

def expand(q):
    out = llm.invoke(
        "Fix the spelling of this question, then write 2 alternative phrasings "
        "using synonyms. Return one per line, nothing else.\n\nQuestion: " + q
    ).content
    lines = [l.strip() for l in out.split("\n") if l.strip()][:3]
    return lines or [q]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [8]:
history = []   

def ask(question):
    search_q = question
    if history:
        past = "\n".join(f"User: {q}\nAI: {a}" for q, a in history[-3:])
        search_q = llm.invoke(
            f"Chat history:\n{past}\n\n"
            f"Rewrite this new question so it makes sense on its own. "
            f"Return only the question.\n\nNew question: {question}"
        ).content

    # Multi-query retrieval (typos + synonyms)
    variants = expand(search_q)
    candidates = {}
    for v in variants:
        for d in vectorstore.similarity_search(v, k=6):
            candidates[d.page_content] = d      # dict removes duplicate chunks

    # Rerank and keep the best TOP_K
    found = rerank(variants[0], list(candidates.values()))

    # Answer from those chunks
    answer = (prompt | llm | StrOutputParser()).invoke(
        {"context": format_docs(found), "question": variants[0]}
    )

    # Sources: (file name, page number)
    sources = sorted({(Path(d.metadata["source"]).name, d.metadata["page"] + 1) for d in found})

    history.append((question, answer))
    return answer, sources

In [9]:
while True:
    q = input("\nAsk a question (or 'quit'): ").strip()
    if q.lower() in {"quit", "exit", "q"}:
        break
    if not q:
        continue
    print("\nMe:", q)
    answer, sources = ask(q)
    print("\nGemma:", answer)
    print("Sources:")
    for name, page in sources:
        print(f"  {name}, page {page}")


Me: hi

Gemma: I don't know.
Sources:
  MLF Lecture 1.pdf, page 43
  MLF Lecture 3.pdf, page 26
  MLF Lecture 3.pdf, page 31
  MLF Lecture 3.pdf, page 37

Me: what is linear regreesion

Gemma: Linear Regression is a supervised learning algorithm used to predict a continuous numerical value.
Sources:
  MLF Lecture 1.pdf, page 19
  MLF Lecture 5.pdf, page 3
  MLF Lecture 5.pdf, page 4
  MLF Lecture 5.pdf, page 5

Me: SGD?

Gemma: I don't know.
Sources:
  MLF Lecture 1.pdf, page 1
  MLF Lecture 1.pdf, page 3
  MLF Lecture 1.pdf, page 33
  MLF Lecture 1.pdf, page 40

Me: KNN?

Gemma: K-Nearest Neighbors (KNN) is a powerful classification algorithm used in pattern recognition and is one of the top data mining algorithms used today. It is a non-parametric lazy learning algorithm (an Instance based Learning method) that stores all available cases and classifies new cases based on a similarity measure, such as a distance function.
Sources:
  MLF Lecture 2 .pdf, page 4
  MLF Lecture 2 .pdf, pa